<div style="direction: rtl; white-space: normal; line-height: 1;">
# Phase 2 — Website Knowledge Ingestion V2

## Step 1 — Website Discovery

هدف این مرحله، کشف تمام مقالات وب‌سایت بدون محدودیت نویسنده است.

در این مرحله فقط لینک مقالات شناسایی و بررسی می‌شوند و هنوز هیچ مقاله‌ای دانلود یا پردازش نمی‌شود.

### اهداف

- شناسایی روش دریافت فهرست تمام مقالات سایت
- پیمایش تمام صفحات Blog
- استخراج لینک‌های `/Blog/Detail/`
- حذف لینک‌های تکراری
- استخراج شناسه مقاله
- ثبت تعداد صفحات و مقالات کشف‌شده
- ایجاد Baseline برای Website Dataset فاز ۲

### نکته

عدد مقالات فاز ۱ یا نتایج قبلی فقط به‌عنوان Baseline استفاده می‌شوند و تعداد مقالات در کد Hard-code نخواهد شد.
</div>

## بررسی روش کشف تمام مقالات سایت

در فاز ۱، دریافت مقالات بر اساس شناسه نویسنده انجام می‌شد.

در فاز ۲ باید محدودیت نویسنده حذف شود و تمام مقالات Blog سایت کشف شوند.

هدف این مرحله پیدا کردن Endpoint یا مکانیزمی است که فهرست کامل Blog را صفحه‌بندی می‌کند.

در این مرحله فقط Discovery بررسی می‌شود و هنوز هیچ مقاله‌ای دانلود نمی‌شود.

<div style="direction: rtl; white-space: normal; line-height: 1;">

</div>

In [7]:
import requests

URL = "https://www.fintelligence.ir/Blog/GetBlogSearchResult"

payload = {
    "PageSize": "12",
    "PageNo": 1,
    "SearchKeyword": "",
    "SelectedBlogCategoryIDs": [],
    "SelectedAgeRangeIDs": [],
    "SelectedBlogContentTypeIDs": [],
    "SelectedBlogTagIDs": [],
    "SelectedBlogPostSortTypeID": "1",
}

response = requests.post(
    URL,
    json=payload,
    timeout=30,
)

print("Status:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))
print(response.text[:2000])

Status: 200
Content-Type: application/json; charset=utf-8
{
  "html": "    <li>\r\n<div class=\"blog-post-list-item1\">\r\n    <figure class=\"blog-post-list-item-image-area\">\r\n        <a href=\"/Blog/Detail/3E933D80/%D9%85%D8%B9%D8%B1%D9%81%DB%8C-%DA%A9%D8%AA%D8%A7%D8%A8-%D9%BE%D9%88%D9%84-%D9%88%D9%84%DB%8C-%D8%AA%D8%A7-%D8%A7%D8%A8%D8%AF-%D9%86%DB%8C%D8%B3%D8%AA\" title=\"\" target=\"_blank\" class=\"blog-post-list-item-image\">\r\n            <img src=\"https://cdn.fintelligence.ir/Uploaded/Blog/Image/8b473e21-984e-4e34-ae35-1b4b42c596aa.jpg\" alt=\"معرفی کتاب شعر &#171;پول ولی تا ابد نیست&#187;\" />\r\n            \r\n        </a>\r\n\r\n    </figure>\r\n    <div class=\"blog-post-list-item-details-area\">\r\n        <h2 class=\"blog-post-list-item-title\">\r\n            <a href=\"/Blog/Detail/3E933D80/%D9%85%D8%B9%D8%B1%D9%81%DB%8C-%DA%A9%D8%AA%D8%A7%D8%A8-%D9%BE%D9%88%D9%84-%D9%88%D9%84%DB%8C-%D8%AA%D8%A7-%D8%A7%D8%A8%D8%AF-%D9%86%DB%8C%D8%B3%D8%AA\" target=\"_blank\" title=

## استخراج لینک‌های مقالات از صفحه اول

در این مرحله HTML برگشتی از Endpoint بررسی می‌شود و فقط لینک‌های مربوط به صفحات مقاله استخراج می‌شوند.

هنوز Pagination اجرا نمی‌شود و فقط صفحه اول برای صحت ساختار بررسی خواهد شد.

In [8]:
from bs4 import BeautifulSoup

data = response.json()
html = data.get("html", "")

soup = BeautifulSoup(html, "html.parser")

links = []

for a in soup.find_all("a", href=True):
    href = a["href"]

    if "/Blog/Detail/" in href:
        links.append(href)

unique_links = sorted(set(links))

print("All article links found:", len(links))
print("Unique article links:", len(unique_links))

for link in unique_links:
    print(link)

All article links found: 36
Unique article links: 12
/Blog/Detail/18EBA5C7/%D8%A2%D9%85%D8%A7%D8%B1%DB%8C-%D8%AA%DA%A9%D8%A7%D9%86-%D8%AF%D9%87%D9%86%D8%AF%D9%87-%D8%A7%D8%B2-%D8%A2%D8%AE%D8%B1%DB%8C%D9%86-%D9%88%D8%B6%D8%B9%DB%8C%D8%AA-%D9%81%D9%82%D8%B1-%D9%88-%D9%86%D8%A7%D8%A7%D9%85%D9%86%DB%8C-%D8%BA%D8%B0%D8%A7%DB%8C%DB%8C-%D8%AF%D8%B1-%D8%A7%DB%8C%D8%B1%D8%A7%D9%86
/Blog/Detail/3E933D80/%D9%85%D8%B9%D8%B1%D9%81%DB%8C-%DA%A9%D8%AA%D8%A7%D8%A8-%D9%BE%D9%88%D9%84-%D9%88%D9%84%DB%8C-%D8%AA%D8%A7-%D8%A7%D8%A8%D8%AF-%D9%86%DB%8C%D8%B3%D8%AA
/Blog/Detail/596080EF/%D8%B3%D9%86%D8%AC%D8%B4-%D8%AA%D8%A7%D8%B2%D9%87-%D9%88%D8%B6%D8%B9%DB%8C%D8%AA-%D8%AF%D8%A7%D9%86%D8%B4-%D9%85%D8%A7%D9%84%DB%8C-%D8%AF%D8%B1-%D8%A7%D8%AA%D8%AD%D8%A7%D8%AF%DB%8C%D9%87-%D8%A7%D8%B1%D9%88%D9%BE%D8%A7
/Blog/Detail/5D28B5FC/%D8%B3%D9%88%D8%A7%D8%AF-%D9%85%D8%A7%D9%84%DB%8C-%D8%A8%D8%B1%D8%A7%DB%8C-%D8%AE%D8%A7%D9%86%D9%85%D9%87%D8%A7-%D9%85%D9%87%D8%A7%D8%B1%D8%AA%D9%87%D8%A7%DB%8C-%D8%B1%D8%A7%D9%87%D8%A7%D9%8

## تست Pagination

در این مرحله Pagination فقط روی سه صفحه اول بررسی می‌شود.

هدف این است که مشخص شود با تغییر `PageNo`، مقالات متفاوت دریافت می‌شوند و روش Discovery برای پیمایش کل سایت قابل استفاده است.

هنوز Crawl کامل سایت انجام نمی‌شود.

In [9]:
from bs4 import BeautifulSoup
import requests

URL = "https://www.fintelligence.ir/Blog/GetBlogSearchResult"

for page_no in range(1, 4):
    payload = {
        "PageSize": "12",
        "PageNo": page_no,
        "SearchKeyword": "",
        "SelectedBlogCategoryIDs": [],
        "SelectedAgeRangeIDs": [],
        "SelectedBlogContentTypeIDs": [],
        "SelectedBlogTagIDs": [],
        "SelectedBlogPostSortTypeID": "1",
    }

    response = requests.post(
        URL,
        json=payload,
        timeout=30,
    )

    response.raise_for_status()

    html = response.json().get("html", "")
    soup = BeautifulSoup(html, "html.parser")

    links = {
        a["href"]
        for a in soup.find_all("a", href=True)
        if "/Blog/Detail/" in a["href"]
    }

    print(f"Page {page_no}: {len(links)} unique articles")

    for link in sorted(links):
        print(" ", link)

    print("-" * 80)

Page 1: 12 unique articles
  /Blog/Detail/18EBA5C7/%D8%A2%D9%85%D8%A7%D8%B1%DB%8C-%D8%AA%DA%A9%D8%A7%D9%86-%D8%AF%D9%87%D9%86%D8%AF%D9%87-%D8%A7%D8%B2-%D8%A2%D8%AE%D8%B1%DB%8C%D9%86-%D9%88%D8%B6%D8%B9%DB%8C%D8%AA-%D9%81%D9%82%D8%B1-%D9%88-%D9%86%D8%A7%D8%A7%D9%85%D9%86%DB%8C-%D8%BA%D8%B0%D8%A7%DB%8C%DB%8C-%D8%AF%D8%B1-%D8%A7%DB%8C%D8%B1%D8%A7%D9%86
  /Blog/Detail/3E933D80/%D9%85%D8%B9%D8%B1%D9%81%DB%8C-%DA%A9%D8%AA%D8%A7%D8%A8-%D9%BE%D9%88%D9%84-%D9%88%D9%84%DB%8C-%D8%AA%D8%A7-%D8%A7%D8%A8%D8%AF-%D9%86%DB%8C%D8%B3%D8%AA
  /Blog/Detail/596080EF/%D8%B3%D9%86%D8%AC%D8%B4-%D8%AA%D8%A7%D8%B2%D9%87-%D9%88%D8%B6%D8%B9%DB%8C%D8%AA-%D8%AF%D8%A7%D9%86%D8%B4-%D9%85%D8%A7%D9%84%DB%8C-%D8%AF%D8%B1-%D8%A7%D8%AA%D8%AD%D8%A7%D8%AF%DB%8C%D9%87-%D8%A7%D8%B1%D9%88%D9%BE%D8%A7
  /Blog/Detail/5D28B5FC/%D8%B3%D9%88%D8%A7%D8%AF-%D9%85%D8%A7%D9%84%DB%8C-%D8%A8%D8%B1%D8%A7%DB%8C-%D8%AE%D8%A7%D9%86%D9%85%D9%87%D8%A7-%D9%85%D9%87%D8%A7%D8%B1%D8%AA%D9%87%D8%A7%DB%8C-%D8%B1%D8%A7%D9%87%D8%A7%D9%86%D8%AF%D8%A7%D8%B

## استخراج شناسه پایدار مقاله

در این مرحله `Article ID` از مسیر URL استخراج می‌شود.

هدف این است که شناسایی مقاله فقط به عنوان یا Slug وابسته نباشد و برای Deduplication و Versioning یک شناسه پایدار داشته باشیم.

In [10]:
def extract_article_id(url):
    parts = url.strip("/").split("/")

    if len(parts) >= 3 and parts[0] == "Blog" and parts[1] == "Detail":
        return parts[2]

    return None


sample_links = sorted(unique_links)

for link in sample_links:
    article_id = extract_article_id(link)
    print(article_id, "->", link)

18EBA5C7 -> /Blog/Detail/18EBA5C7/%D8%A2%D9%85%D8%A7%D8%B1%DB%8C-%D8%AA%DA%A9%D8%A7%D9%86-%D8%AF%D9%87%D9%86%D8%AF%D9%87-%D8%A7%D8%B2-%D8%A2%D8%AE%D8%B1%DB%8C%D9%86-%D9%88%D8%B6%D8%B9%DB%8C%D8%AA-%D9%81%D9%82%D8%B1-%D9%88-%D9%86%D8%A7%D8%A7%D9%85%D9%86%DB%8C-%D8%BA%D8%B0%D8%A7%DB%8C%DB%8C-%D8%AF%D8%B1-%D8%A7%DB%8C%D8%B1%D8%A7%D9%86
3E933D80 -> /Blog/Detail/3E933D80/%D9%85%D8%B9%D8%B1%D9%81%DB%8C-%DA%A9%D8%AA%D8%A7%D8%A8-%D9%BE%D9%88%D9%84-%D9%88%D9%84%DB%8C-%D8%AA%D8%A7-%D8%A7%D8%A8%D8%AF-%D9%86%DB%8C%D8%B3%D8%AA
596080EF -> /Blog/Detail/596080EF/%D8%B3%D9%86%D8%AC%D8%B4-%D8%AA%D8%A7%D8%B2%D9%87-%D9%88%D8%B6%D8%B9%DB%8C%D8%AA-%D8%AF%D8%A7%D9%86%D8%B4-%D9%85%D8%A7%D9%84%DB%8C-%D8%AF%D8%B1-%D8%A7%D8%AA%D8%AD%D8%A7%D8%AF%DB%8C%D9%87-%D8%A7%D8%B1%D9%88%D9%BE%D8%A7
5D28B5FC -> /Blog/Detail/5D28B5FC/%D8%B3%D9%88%D8%A7%D8%AF-%D9%85%D8%A7%D9%84%DB%8C-%D8%A8%D8%B1%D8%A7%DB%8C-%D8%AE%D8%A7%D9%86%D9%85%D9%87%D8%A7-%D9%85%D9%87%D8%A7%D8%B1%D8%AA%D9%87%D8%A7%DB%8C-%D8%B1%D8%A7%D9%87%D8%A7%D9%86%D8%

## ساخت URL کامل و Canonical

در این مرحله لینک‌های نسبی مقاله به URL کامل تبدیل می‌شوند.

هدف این است که برای هر مقاله یک URL استاندارد داشته باشیم و در مراحل بعدی Manifest و Deduplication به مسیر نسبی وابسته نباشند.

In [11]:
from urllib.parse import urljoin

BASE_URL = "https://www.fintelligence.ir"

for link in sample_links:
    article_id = extract_article_id(link)
    absolute_url = urljoin(BASE_URL, link)

    canonical_url = f"{BASE_URL}/Blog/Detail/{article_id}"

    print("Article ID:", article_id)
    print("Absolute URL:", absolute_url)
    print("Canonical URL:", canonical_url)
    print("-" * 80)

Article ID: 18EBA5C7
Absolute URL: https://www.fintelligence.ir/Blog/Detail/18EBA5C7/%D8%A2%D9%85%D8%A7%D8%B1%DB%8C-%D8%AA%DA%A9%D8%A7%D9%86-%D8%AF%D9%87%D9%86%D8%AF%D9%87-%D8%A7%D8%B2-%D8%A2%D8%AE%D8%B1%DB%8C%D9%86-%D9%88%D8%B6%D8%B9%DB%8C%D8%AA-%D9%81%D9%82%D8%B1-%D9%88-%D9%86%D8%A7%D8%A7%D9%85%D9%86%DB%8C-%D8%BA%D8%B0%D8%A7%DB%8C%DB%8C-%D8%AF%D8%B1-%D8%A7%DB%8C%D8%B1%D8%A7%D9%86
Canonical URL: https://www.fintelligence.ir/Blog/Detail/18EBA5C7
--------------------------------------------------------------------------------
Article ID: 3E933D80
Absolute URL: https://www.fintelligence.ir/Blog/Detail/3E933D80/%D9%85%D8%B9%D8%B1%D9%81%DB%8C-%DA%A9%D8%AA%D8%A7%D8%A8-%D9%BE%D9%88%D9%84-%D9%88%D9%84%DB%8C-%D8%AA%D8%A7-%D8%A7%D8%A8%D8%AF-%D9%86%DB%8C%D8%B3%D8%AA
Canonical URL: https://www.fintelligence.ir/Blog/Detail/3E933D80
--------------------------------------------------------------------------------
Article ID: 596080EF
Absolute URL: https://www.fintelligence.ir/Blog/Detail/596080EF/%D

## بررسی نقطه پایان Pagination

در این مرحله رفتار Endpoint در صفحات انتهایی بررسی می‌شود.

هدف این است که Crawler بدون Hard-code کردن تعداد صفحات، بتواند پایان Blog را به‌صورت خودکار تشخیص دهد.

In [12]:
from bs4 import BeautifulSoup
import requests

URL = "https://www.fintelligence.ir/Blog/GetBlogSearchResult"

for page_no in range(40, 56):
    payload = {
        "PageSize": "12",
        "PageNo": page_no,
        "SearchKeyword": "",
        "SelectedBlogCategoryIDs": [],
        "SelectedAgeRangeIDs": [],
        "SelectedBlogContentTypeIDs": [],
        "SelectedBlogTagIDs": [],
        "SelectedBlogPostSortTypeID": "1",
    }

    response = requests.post(
        URL,
        json=payload,
        timeout=30,
    )

    response.raise_for_status()

    html = response.json().get("html", "")
    soup = BeautifulSoup(html, "html.parser")

    links = {
        a["href"]
        for a in soup.find_all("a", href=True)
        if "/Blog/Detail/" in a["href"]
    }

    print(
        f"Page {page_no}: "
        f"html_length={len(html)}, "
        f"unique_articles={len(links)}"
    )

Page 40: html_length=22355, unique_articles=12
Page 41: html_length=24776, unique_articles=12
Page 42: html_length=22942, unique_articles=12
Page 43: html_length=22230, unique_articles=12
Page 44: html_length=21407, unique_articles=12
Page 45: html_length=1832, unique_articles=1
Page 46: html_length=0, unique_articles=0
Page 47: html_length=0, unique_articles=0
Page 48: html_length=0, unique_articles=0
Page 49: html_length=0, unique_articles=0
Page 50: html_length=0, unique_articles=0
Page 51: html_length=0, unique_articles=0
Page 52: html_length=0, unique_articles=0
Page 53: html_length=0, unique_articles=0
Page 54: html_length=0, unique_articles=0
Page 55: html_length=0, unique_articles=0


<div style="direction: rtl; white-space: normal; line-height: 1;">
نتیجه این مرحله:

صفحات 1 تا 44 هر کدام ۱۲ مقاله دارند.
صفحه 45 فقط ۱ مقاله دارد.
از صفحه 46 به بعد خروجی خالی است.
بنابراین تعداد فعلی مقالات کشف‌شده برابر است با:
44 × 12 + 1 = 529
</div>

## ساخت تابع کامل Website Discovery

در این مرحله منطق کشف تمام مقالات به یک تابع مستقل تبدیل می‌شود.

Crawler باید:

- از صفحه اول شروع کند.
- صفحات را یکی‌یکی بخواند.
- لینک‌های مقاله را استخراج کند.
- در اولین صفحه بدون مقاله متوقف شود.
- لینک‌های تکراری را حذف کند.
- تعداد صفحات و مقالات را گزارش دهد.

تعداد صفحات یا تعداد مقالات در کد Hard-code نمی‌شود.